In [145]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression, SGDRegressor,LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False


In [146]:
data = {
    '수치형_특징': [10, 20, 30, 40, 50],
    '범주형_특징': ['A', 'B', 'A', 'C', 'B'],
    '그대로_유지': [1, 0, 1, 0, 1] # 정답
}
df = pd.DataFrame(data)
df

,수치형_특징,범주형_특징,그대로_유지
0,10,A,1
1,20,B,0
2,30,A,1
3,40,C,0
4,50,B,1


In [147]:
x_data = df.iloc[:, :-1].values
y_data = df.iloc[:, -1].values

In [148]:
# 파이프라인을 통해 심플하게 처리 가능
# ordinalEncoder , logisticRegression
# model_pipe = make_pipeline(OrdinalEncoder(),LogisticRegression()) 모델명만 나열하면 이름이 소문자로 자동으로 생성 됨
model_pipe = Pipeline([('encode', OrdinalEncoder()), # OrdinalEncoder 숫자로 변형됨
                        ('logi',LogisticRegression(max_iter=500))]) # 리스트의 튜플로 (모델에 대한 이름(맘대로), 실제 사용할 모델)

model_pipe.fit(x_data, y_data) # x_data 인코딩하여 학습함

,steps,"[('encode', ...), ('logi', ...)]"
,transform_input,None
,memory,None
,verbose,False
,categories,'auto'
,dtype,<class 'numpy.float64'>
,handle_unknown,'error'
,unknown_value,None
,encoded_missing_value,nan
,min_frequency,None
,max_categories,None


In [149]:
model_pipe.predict([[10,'A']]) # 인코딩 숫자로 변환해서 예측값 얻어옴

array([1])

In [150]:
enc = model_pipe.named_steps['encode']
enc.categories_ # 숫자는 그대로, 'A', 'B', 'C'가 인코딩 됨
### 모든 특성 데이터 일괄 라벨 인코더 적용

[array([10, 20, 30, 40, 50], dtype=object),
 array(['A', 'B', 'C'], dtype=object)]

###  ColumnTransformer 이용

In [151]:
data = {
    '수치형_특징': [10, 20, 30, 40, 50],
    '범주형_특징': ['A', 'B', 'A', 'C', 'B'],
    '그대로_유지': [1, 0, 1, 0, 1] # 정답
}
df = pd.DataFrame(data)
df

,수치형_특징,범주형_특징,그대로_유지
0,10,A,1
1,20,B,0
2,30,A,1
3,40,C,0
4,50,B,1


### 특성 데이터 라벨이 데이터 프레임인 경우

In [152]:
x_data = df.iloc[:, :-1]
y_data = df.iloc[:, -1]

In [153]:
# column_preprocessor = ColumnTransformer([('enc', OrdinalEncoder(),['범주형_특징'])],
#                                         remainder='passthrough') 
# model_list = [('nan', SimpleImputer(strategy='median'),['난컬럼']),
#                 ('scale', StandardScaler(),['수치형_특징']),
#                 ('enc', OrdinalEncoder(),['범주형_특징'])] 
# column_preprocessor = ColumnTransformer(model_list, remainder='passthrough')  nan 컬럼이 있을 시 사용

model_list = [('scale', StandardScaler(),['수치형_특징']),
                ('enc', OrdinalEncoder(),['범주형_특징'])] # 수치형 특징으로 스케일링, 범주형 특징으로 모델링

column_preprocessor = ColumnTransformer(model_list, remainder='passthrough') 
# 리스트의 튜플로 줌, 대상 지정 가능 , remainder='passthrough' 범주형_특징 외 나머지는 통과

model_cpipe = Pipeline( [ ('ct', column_preprocessor),
                        ('logi',LogisticRegression(max_iter=500)) ] )

model_cpipe.fit(x_data, y_data)

,steps,"[('ct', ...), ('logi', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('scale', ...), ('enc', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [154]:
xd = pd.DataFrame({'수치형_특징':[10],'범주형_특징':['A']}) # 데이터프레임이라 오류남, 범주형_특징이 뭔지 알려줘야함
model_cpipe.predict(xd)

array([1])

### 특성 데이터 라벨이 넘파이인 경우

In [155]:
x_data = df.iloc[:, :-1].values
y_data = df.iloc[:, -1].values # 데이터 프레임이라 두번째 컬럼이 OrdinalEncoder 대상이라 [1]로 줘야함

In [156]:
x_data

array([[10, 'A'],
       [20, 'B'],
       [30, 'A'],
       [40, 'C'],
       [50, 'B']], dtype=object)

In [157]:
column_preprocessor = ColumnTransformer([('enc', OrdinalEncoder(),[1])],
                                        remainder='passthrough') 
# 리스트의 튜플로 줌, 대상 지정 가능 , remainder='passthrough' 범주형_특징 외 나머지는 통과

model_cpipe = Pipeline( [ ('ct', column_preprocessor),
                        ('logi',LogisticRegression(max_iter=500)) ] )

model_cpipe.fit(x_data, y_data)

,steps,"[('ct', ...), ('logi', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('enc', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [158]:
model_cpipe.predict([[10,'A']])

array([1])

```text
### 퀴즈
특성 데이터: 년식, 종류, 연비, 마력, 토크, 연료
라벨 :  가격
년식    종류    연비    마력    토그    연료
2015,  준중형,  12.3,   204,    27,    가솔린

In [200]:
hdf = pd.read_excel('data/hyundaiCar.xlsx')
hdf

,가격,년식,종류,연비,마력,토크,연료,하이브리드,배기량,중량,변속기
0,1885,2015,준중형,11.8,172,21.0,가솔린,0,1999,1300,자동
1,2190,2015,준중형,12.3,204,27.0,가솔린,0,1591,1300,자동
2,1135,2015,소형,15.0,100,13.6,가솔린,0,1368,1035,수동
3,1645,2014,소형,14.0,140,17.0,가솔린,0,1591,1090,자동
4,1960,2015,대형,9.6,175,46.0,디젤,0,2497,1990,자동
...,...,...,...,...,...,...,...,...,...,...,...
66,3802,2015,중형,8.5,290,34.8,가솔린,0,3342,1901,자동
67,1270,2012,소형,13.3,108,13.9,가솔린,0,1396,1040,자동
68,2430,2015,준중형,12.8,186,41.0,디젤,0,1995,1665,자동
69,2870,2015,중형,17.7,156,19.3,가솔린,1,1999,1585,자동


In [201]:
x_data = hdf.iloc[:, 1:7]
y_data = hdf.iloc[:, 0]

In [202]:
x_data

,년식,종류,연비,마력,토크,연료
0,2015,준중형,11.8,172,21.0,가솔린
1,2015,준중형,12.3,204,27.0,가솔린
2,2015,소형,15.0,100,13.6,가솔린
3,2014,소형,14.0,140,17.0,가솔린
4,2015,대형,9.6,175,46.0,디젤
...,...,...,...,...,...,...
66,2015,중형,8.5,290,34.8,가솔린
67,2012,소형,13.3,108,13.9,가솔린
68,2015,준중형,12.8,186,41.0,디젤
69,2015,중형,17.7,156,19.3,가솔린


In [203]:
y_data

0     1885
1     2190
2     1135
3     1645
4     1960
      ... 
66    3802
67    1270
68    2430
69    2870
70    3254
Name: 가격, Length: 71, dtype: int64

In [204]:
h_list = [('scale',StandardScaler(),['년식','연비','마력','토크']),
        ('enc',OneHotEncoder(handle_unknown='ignore'),['종류','연료'])]

In [205]:
column_preprocessor = ColumnTransformer(h_list, remainder='passthrough') 

In [206]:
model_cpipe = Pipeline( [ ('ct', column_preprocessor),
                        ('reg', LinearRegression()) ] )

model_cpipe.fit(x_data, y_data)

,steps,"[('ct', ...), ('reg', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('scale', ...), ('enc', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [208]:
xd = pd.DataFrame({'년식':[2015],'종류':['준중형'],'연비':[12.3],'마력':[204],'토크':[27],'연료':['가솔린']})
model_cpipe.predict(xd)

array([2820.31702944])

---